### Package

In [273]:
import pandas as pd
import unicodedata
import numpy as np
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout,SimpleRNN,Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

### preposising

In [274]:
df=pd.read_csv(r"C:\Users\Admin\Downloads\twitter.csv")
df.head()

,labels,text
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...


In [275]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75680 entries, 0 to 75679
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   labels  75680 non-null  object
 1   text    74994 non-null  object
dtypes: object(2)
memory usage: 1.2+ MB


In [276]:
df['labels'].value_counts()

labels
Negative      22808
Positive      21108
Neutral       18603
Irrelevant    13161
Name: count, dtype: int64

In [277]:
df.isnull().sum()

labels      0
text      686
dtype: int64

In [278]:
df.dropna( inplace=True)

In [279]:
df.drop_duplicates( inplace=True)

### Text Prep

In [280]:
df['text'].head(20)

0     I am coming to the borders and I will kill you...
1     im getting on borderlands and i will kill you ...
2     im coming on borderlands and i will murder you...
3     im getting on borderlands 2 and i will murder ...
4     im getting into borderlands and i can murder y...
5     So I spent a few hours making something for fu...
6     So I spent a couple of hours doing something f...
7     So I spent a few hours doing something for fun...
8     So I spent a few hours making something for fu...
9     2010 So I spent a few hours making something f...
10                                                  was
11    Rock-Hard La Varlope, RARE & POWERFUL, HANDSOM...
12    Rock-Hard La Varlope, RARE & POWERFUL, HANDSOM...
13    Rock-Hard La Varlope, RARE & POWERFUL, HANDSOM...
14    Rock-Hard La Vita, RARE BUT POWERFUL, HANDSOME...
15    Live Rock - Hard music La la Varlope, RARE & t...
16    I-Hard like me, RARE LONDON DE, HANDSOME 2011,...
17    that was the first borderlands session in 

In [281]:

def process_text(text):
    # don't  don’t
    text = unicodedata.normalize("NFKD", text)
    text = text.encode('ascii', 'ignore').decode("utf-8","ignore")

    text = text.lower()
    text = re.sub(r"https?//\S+|www\.\S+|\b\w+\.(com|ly|net|org|eg)\S*|\b\w+\.\w+\.(com|ly|net|org|eg)\S*", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", "\1", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z'\s]", ' ', text)
    text = re.sub(r"\s+", " ", text)

    words = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    lemm = WordNetLemmatizer()

    words = [lemm.lemmatize(word) for word in words if word not in stop_words]
    words = [word for word in words if len(word) > 2]
    
    
    uni = np.unique(words, return_index=True)[1]
    final_text = np.array(words)[np.sort(uni)].tolist()

    return final_text

In [282]:
df['clean_text'] = df['text'].apply(process_text)


In [283]:
x=df['clean_text']
y=df['labels']
y=y.map({"Negative": 0, "Positive": 1, "Neutral": 2, "Irrelevant": 3})
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [284]:
max_vocab = 20000
tok = Tokenizer(num_words=max_vocab)
tok.fit_on_texts(x_train)
word_index = tok.word_index
num_word=len(word_index) 
x_train = tok.texts_to_sequences(x_train)
x_test = tok.texts_to_sequences(x_test)
num_word

24889

In [285]:
maxlen=0
for i in x_train:
        maxlen=max(maxlen,len(i))
x_train = pad_sequences(x_train, padding='post', maxlen=maxlen)
x_test = pad_sequences(x_test, padding='post', maxlen=maxlen)

In [286]:
model = Sequential([
    Input(shape=(maxlen,)),
    Embedding(num_word+1, 100),
    LSTM(100),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')
])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 42, 100)        │     2,489,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         3,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,572,764 (9.81 MB)

 Trainable params: 2,572,764 (9.81 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=10, validation_split=0.1)

<Sequential name=sequential_4, built=True>

In [288]:
# import unicodedata
# import numpy as np

# def process_text(text):
#     # don’t
#     text=unicodedata.normalize("NFKD",text)
#     text=text.encode('ascii','ignore').decode("utf-8","ignore")
#     text = re.sub(r"\b[A-Z]{2,}\b", "", text)
#     text=text.lower()
#     text=re.sub(r'\b(\w+)(?:\W+\1\b)+','',text)
#     text = re.sub(r"\s+", " ", text)
#     text = re.sub(r"\d+", " ", text)
#     text = re.sub(r"[^\w\s]", " ", text)
#     text = re.sub(r"#(\w+)", r"\1", text)
#     text = re.sub(r"@\w+", "", text)
#     text=re.sub(r"<.*?>", "", text)
    
#     text=re.sub(r"\w+@gmail\.\w+",'',text)
#     text = re.sub(r"https?://\S+|www\.\S+|\b\w?+\.(com|org|net|ly|eg)\S*", "", text)
    
#     text = re.sub(r"\n+", " ", text)
#     # 6th__
#     text = re.sub(r"[_\d]+", " ", text)
#     text = re.sub(r"\b\d+[a-zA-Z]*_+\b", " ", text)
    
    
#     words = word_tokenize(text)
#     stop_words = set(stopwords.words("english"))
#     lemm=WordNetLemmatizer()
#     words = [lemm.lemmatize(word) for word in words if word not in stop_words]
#     words = [word for word in words if len(word) > 3]
#     uni=np.unique(words,return_index=True)[1]
#     final_text=np.array(words)[np.sort(uni)].tolist()
#     return final_text